In [1]:
import pickle
from pathlib import Path

all_results: list[Path] = [
  p.absolute()
  for p in Path("consistency_reports").rglob("*")
  if p.is_file() and p.suffix.lower() == ".pkl"
]

results: dict[str, list[dict]] = {}
for result_path in all_results:

  with open(result_path, "rb") as f:
    result: list = pickle.load(f)
    results[result_path.stem] = result

print(f"Loaded {sum(1 for x in results.values() for y in x)} results from {len(all_results)} files.")

Loaded 8 results from 1 files.


In [5]:
from datetime import datetime
import platform

import numpy as np
import pandas as pd

from birdnet.acoustic_models.inference.scores.prediction_result import PredictionResult
from birdnet.helper import get_hash
from birdnet.local_data import get_package_version


def get_sorted_probs(prediction_result: PredictionResult) -> np.ndarray:
  sort_idx = np.argsort(prediction_result.species_ids, axis=-1)
  sorted_probs = np.take_along_axis(prediction_result.species_probs, sort_idx, axis=-1)
  return sorted_probs


def create_report(name: str, reference: dict, results: list[dict], thresholds: list[float]):
  report = []
  assert reference["backend"] == "pb" and reference["device"] == "CPU"
  now = datetime.now()
  now_time = now.strftime("%Y/%m/%d %I:%M:%S %p")

  meta = {
    "run_name": "",
    "setup_hash": "",
    "version": get_package_version(),
    "python": f"{platform.python_version()} {platform.python_implementation()}",
    "hw_host": platform.platform(),
    "hw_cpu": platform.processor(),
  }
  platform_hash = f"{meta['python']}-{meta['hw_host']}-{meta['hw_cpu']}"
  hash_digest = get_hash(platform_hash)[:5]
  meta["setup_hash"] = hash_digest

  ref = get_sorted_probs(reference["result"])
  for res in results:
    for threshold in thresholds:
      mask = ref >= threshold
      scores = get_sorted_probs(res["result"])
      prob_diff = np.abs(ref - scores)
      masked_diff = prob_diff[mask]
      max_diff = np.max(masked_diff)
      mean_diff = np.mean(masked_diff)
      min_diff = np.min(masked_diff)
      std_diff = np.std(masked_diff)
      q1 = np.percentile(masked_diff, 25)
      median_diff = np.median(masked_diff)
      q3 = np.percentile(masked_diff, 75)
      n_segments = ref.shape[1]
      n_species = ref.shape[2]
      n_values = masked_diff.size
      total_values = prob_diff.size

      report_entry = meta | {
        "date": now_time,
        "backend": res["backend"],
        "precision": res["precision"],
        "n_species": n_species,
        "device": res["device"],
        # "init_time_s": res["init_time_s"],
        # "prediction_time_s": res["prediction_time_s"],
        # "total_time_s": res["total_time_s"],
        "compare_threshold": threshold,
        "n_segments": n_segments,
        "total_values": total_values,
        "n_values": n_values,
        "n_values_percent": round(n_values / total_values * 100, 2),
        "mean_diff": mean_diff,
        "std_diff": std_diff,
        "min_diff": min_diff,
        "max_diff": max_diff,
        "q1_diff": q1,
        "q2_diff": median_diff,
        "q3_diff": q3,
      }
      report_entry["run_name"] = res["run_name"]
      report.append(report_entry)
  df_report = pd.DataFrame(report)

  report_path = Path("consistency_reports") / f"{name}.csv"
  df_report.to_csv(report_path, index=False)
  return df_report



In [6]:
# Internal comparison

def internal_comparison(report_name: str, report: list[dict], thresholds: list[float]):
  create_report(f"{report_name}_interal", report[0], report[1:], thresholds)

for name, res_list in results.items():
  internal_comparison(name, res_list, thresholds=[0, 0.001, 0.01, 0.1, 0.2, 0.3])
